# Tier 2 Model: XGBoost on TF-IDF

This notebook trains an XGBoost model on the same TF-IDF features and train/test split as the Logistic Regression baseline, then explains it with post-hoc methods because, like the Random Forest, a boosted ensemble of many trees is not transparent by design.

The explanation steps follow the course's interpretation pipeline: global importance (PFI / SHAP) → feature effects (PDP) → heterogeneity (ICE) → individual decisions (LIME / SHAP).

## 1.Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("punkt")
nltk.download("punkt_tab")

from IPython.display import display
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, recall_score, accuracy_score, roc_auc_score
)
from sklearn.inspection import PartialDependenceDisplay

from lime.lime_text import LimeTextExplainer
from lime.submodular_pick import SubmodularPick
import shap

from preprocessing import (
    preprocess_text, get_tfidf_features,
    get_train_test_split, select_top_k_terms, RANDOM_STATE
)

## 2. Load Cleaned Dataset

In [ ]:
df = pd.read_csv("Dataset/fake_news_dataset_cleaned.csv")
df[["title", "text", "label", "label_encoded"]].head()

## 3. Text Preprocessing (NLP Pipeline)

In [ ]:
df["full_text"] = df["title"] + " " + df["text"]
df["processed_text"] = df["full_text"].apply(preprocess_text)
df[["full_text", "processed_text"]].head()

Quick sanity check: compare one raw vs. processed example side by side.

In [ ]:
print("RAW:\n", df["full_text"].iloc[0][:300])
print("\nPROCESSED:\n", df["processed_text"].iloc[0][:300])

## 4. TF-IDF Vectorization

In [ ]:
X, y, tfidf = get_tfidf_features(df)
feature_names = tfidf.get_feature_names_out()

print("TF-IDF matrix shape:", X.shape)

## 5. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = get_train_test_split(X, y)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain class balance:\n", y_train.value_counts(normalize=True).round(3))
print("\nTest class balance:\n", y_test.value_counts(normalize=True).round(3))

### 5.1 Dense Arrays for XGBoost

XGBoost treats the zeros in a sparse matrix as missing values, so the same article can get a different prediction depending on whether it is passed in sparse or dense form. To keep training, evaluation and every explanation consistent, the TF-IDF matrix is converted to a dense array here and the dense version is used from now on.

In [ ]:
X_train_dense = X_train.toarray().astype(np.float32)
X_test_dense = X_test.toarray().astype(np.float32)

## 6. Train XGBoost

XGBoost builds its trees one after another, each correcting the mistakes of the ones before it. Tree depth, the minimum weight of a leaf and the learning rate are limited, to keep the model small, to limit how much it memorizes the training articles, and to keep it fast to explain with TreeSHAP.

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=10,
    tree_method="hist",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_model.fit(X_train_dense, y_train)
y_pred = xgb_model.predict(X_test_dense)

## 7. Validate Predictive Performance

In [ ]:
print(classification_report(y_test, y_pred, target_names=["real", "fake"]))

recall_fake = recall_score(y_test, y_pred, pos_label=1)
accuracy = accuracy_score(y_test, y_pred)
train_recall = recall_score(y_train, xgb_model.predict(X_train_dense), pos_label=1)
auc = roc_auc_score(y_test, xgb_model.predict_proba(X_test_dense)[:, 1])

print(f"Fake-class recall (test):  {recall_fake:.4f}")
print(f"Fake-class recall (train): {train_recall:.4f}   <- large gap vs test = overfitting")
print(f"Accuracy (test): {accuracy:.4f}")
print(f"ROC AUC (test): {auc:.4f}")
print(f"Share of test articles predicted fake: {y_pred.mean():.3f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["real", "fake"]).plot(cmap="Blues")
plt.title("Confusion Matrix: XGBoost")
plt.show()

## 8. Top-K Terms and Multicollinearity Check

Similar with all other models with 867 terms, inspecting every term is impractical: PDP and ICE would need hundreds of plots, and an 867 × 867 correlation matrix cannot be read. We therefore focus on the 30 terms most linked to the fake/real label (K = 30), ranked by a chi-squared score on the training data only.

Chi-squared measures how unevenly a term's weight is split between fake and real articles. A higher score means a stronger link to the label, but it does not show the direction. We use it because it does not depend on any model, so every notebook can share the same list. Since TF-IDF weights are not counts, the scores are used only for ranking, not as significance tests.

In [ ]:
K = 30
top_idx, top_terms, top_scores = select_top_k_terms(X_train, y_train, feature_names, k=K, method="chi2")
pd.DataFrame({"term": top_terms, "chi2_score": top_scores}).head(7)

The table shows the seven terms with the highest chi-squared scores, meaning the terms most strongly associated with the fake/real label in the training data. The terms (economy, particular, significant, model, girl, bring, reflect) are ordinary words rather than sensational or emotionally charged language, and the scores are low and close together (about 0.50 to 0.58), so no single term stands out. The table also does not say whether a term points toward fake or real.

All 30 terms are used in the analysis below, but only 7 are displayed at a time. Human working memory can hold about 7 ± 2 items at once, so limiting each table and plot to about 7 items lets the reader take it in at a glance without being overloaded.

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
sample_idx = rng.choice(X_test_dense.shape[0], size=500, replace=False)
X_sample = X_test_dense[sample_idx]          # 500-row sample for PDP/ICE and SHAP

The correlation heatmap checks whether any of the top-K terms move together across the training articles. Each cell shows the correlation between the TF-IDF weights of two terms, where 1 means they rise and fall together, -1 means one rises when the other falls, and 0 means no relationship. We also search for any pair above 0.5. This matters because the result decides which feature-effect technique we use later: PDP changes one term while holding the others fixed, which is only safe if the terms are not dependent on each other, and related terms can also split importance between them, which would distort PFI.

In [ ]:
X_train_topk = pd.DataFrame(X_train[:, top_idx].toarray(), columns=top_terms)
corr = X_train_topk.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation Between Top-K Terms (training data)")
plt.tight_layout()
plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
high_pairs = pairs[pairs.abs() > 0.5]
if high_pairs.empty:
    print("No pairs of terms with |correlation| > 0.5")
else:
    print("Pairs with |correlation| > 0.5:")
    print(high_pairs.sort_values(key=abs, ascending=False))

The heatmap is almost entirely pale gray, meaning the correlations between different terms are close to 0. The only strong cells are the red diagonal, which is each term compared with itself and is always 1. There are no red or blue cells off the diagonal, so none of the top 30 terms tends to appear together with, or instead of, another. The search for pairs of terms with a correlation above 0.5 also found none. This tells us the terms are close to independent of each other, so PDP can be used without creating unrealistic articles, and PFI should not be splitting importance between related terms. This check covers only the top 30 terms and only linear relationships, so it does not rule out every possible dependence, but it gives no reason to switch to ALE.

## 9. Global Importance (PFI and TreeSHAP)

### 9.1 Permutation Feature Importance (PFI)

Each top-K term is shuffled in the test set and the drop in Fake-class recall is measured (5 repeats). A large drop means the model relies on that term.

In [ ]:
def permutation_recall_drop(model, X_dense, y_true, col_indices, n_repeats=5, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    base = recall_score(y_true, model.predict(X_dense), pos_label=1)
    rows = []
    for col in col_indices:
        original = X_dense[:, col].copy()
        drops = []
        for _ in range(n_repeats):
            X_dense[:, col] = rng.permutation(original)
            drops.append(base - recall_score(y_true, model.predict(X_dense), pos_label=1))
        X_dense[:, col] = original
        rows.append((feature_names[col], np.mean(drops), np.std(drops)))
    result = pd.DataFrame(rows, columns=["term", "mean_recall_drop", "std"])
    return base, result.sort_values("mean_recall_drop", ascending=False)

base_recall, pfi_df = permutation_recall_drop(xgb_model, X_test_dense, y_test, top_idx)
print(f"Recall before shuffling: {base_recall:.4f}")
pfi_df.head(7)

In [ ]:
top_pfi = pfi_df.head(7).iloc[::-1]
plt.figure(figsize=(8, 6))
plt.barh(top_pfi["term"], top_pfi["mean_recall_drop"], xerr=top_pfi["std"], color="steelblue")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("PFI: Drop in Fake-class Recall When Term Is Shuffled")
plt.xlabel("Mean recall drop (± std over repeats)")
plt.tight_layout()
plt.show()

#### Is PFI measuring anything?

As a check, we shuffle 30 random terms from outside the top 30. If their drops are as large as those of the top terms, the PFI ranking cannot be told apart from noise.

In [ ]:
rng_null = np.random.RandomState(RANDOM_STATE)
other_idx = rng_null.choice(np.setdiff1d(np.arange(len(feature_names)), top_idx), 30, replace=False)
_, pfi_null = permutation_recall_drop(xgb_model, X_test_dense, y_test, other_idx)
print(pfi_null["mean_recall_drop"].describe())

### 9.2 TreeSHAP (global view)

SHAP values attribute each prediction to individual terms (baseline + contributions = prediction). They are computed on a random sample of 200 of the 4,000 test articles to keep runtime manageable, then averaged for a global view. For boosted trees, SHAP values are measured in log-odds and not in probability: positive values push toward fake and negative values toward real, and near a probability of 0.5 a log-odds value of 0.1 corresponds to a change of about 2.5 percentage points.

In [ ]:
explainer_shap = shap.TreeExplainer(xgb_model)
X_shap = X_sample[:200]

sv = explainer_shap.shap_values(X_shap, check_additivity=False)

# SHAP's output format differs by version; extract values for the FAKE class (index 1)
if isinstance(sv, list):
    sv_fake = sv[1]
elif sv.ndim == 3:
    sv_fake = sv[:, :, 1]
else:
    sv_fake = sv

ev = explainer_shap.expected_value
base_value = ev[1] if np.ndim(ev) > 0 else ev
print("SHAP values shape:", sv_fake.shape)

In [ ]:
shap.summary_plot(sv_fake, X_shap, feature_names=list(feature_names), max_display=7)

In [ ]:
shap_importance = pd.DataFrame({
    "term": feature_names,
    "mean_abs_shap": np.abs(sv_fake).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

shap_importance.head(7)

## 10. Feature Effects (PDP)

As mentioned above, the top terms are not correlated with each other, so changing one term's value while holding the others fixed will not create unrealistic articles. This means PDP can be used here instead of ALE.

PDP shows the model's average predicted probability of fake as one term's weight increases, so it tells us which direction each term pushes the prediction and how strongly, on average.

In [ ]:
pdp_idx = list(top_idx[:6])

pdp_display = PartialDependenceDisplay.from_estimator(
    xgb_model, X_sample,
    features=pdp_idx,
    feature_names=list(feature_names),
    kind="average",
    grid_resolution=20,
    percentiles=(0, 1),       # TF-IDF is mostly zeros, so the default 5-95% range would collapse to a single value
    n_cols=3,
    random_state=RANDOM_STATE,
)
pdp_display.figure_.set_size_inches(14, 8)
pdp_display.figure_.suptitle("PDP: Average Predicted Probability of Fake vs Term TF-IDF Weight")
plt.tight_layout()
plt.show()

## 11. Heterogeneity (ICE)

ICE draws one line per article instead of only the average, so it shows whether a term affects every article the same way, and crossing or diverging lines mean the effect depends on the other terms in the article.

In [ ]:
ice_display = PartialDependenceDisplay.from_estimator(
    xgb_model, X_sample,
    features=pdp_idx,
    feature_names=list(feature_names),
    kind="both",
    subsample=100,
    grid_resolution=20,
    percentiles=(0, 1),
    n_cols=3,
    random_state=RANDOM_STATE,
)
ice_display.figure_.set_size_inches(14, 8)
ice_display.figure_.suptitle("ICE: Individual Article Curves (dashed line = PDP average)")
plt.tight_layout()
plt.show()

## 12. Individual Decisions (LIME and SHAP)

So far we have looked at why the model makes its decisions as a whole (global). Now we pick specific scenarios and ask why the model made this particular decision. We first use SP-LIME to let the data choose a small set of representative articles, and then look at three cases chosen by outcome: a fake article the model caught correctly, a fake article it missed, and a real article it wrongly flagged as fake. LIME and SHAP are used to show which terms pushed each prediction toward fake or toward real.

In [ ]:
explainer_lime = LimeTextExplainer(class_names=["real", "fake"], random_state=RANDOM_STATE)

def predict_proba_text(texts):
    # XGBoost was trained on dense arrays, so the TF-IDF output is converted to dense here as well
    return xgb_model.predict_proba(tfidf.transform(texts).toarray().astype(np.float32))

test_text = df.loc[y_test.index, "processed_text"].reset_index(drop=True)
y_true_arr = y_test.reset_index(drop=True).to_numpy()
y_pred_arr = np.asarray(y_pred)
proba_fake = xgb_model.predict_proba(X_test_dense)[:, 1]

tp = np.where((y_true_arr == 1) & (y_pred_arr == 1))[0]
fn = np.where((y_true_arr == 1) & (y_pred_arr == 0))[0]
fp = np.where((y_true_arr == 0) & (y_pred_arr == 1))[0]

cases = {
    "True Positive (fake, caught)":      tp[np.argmax(proba_fake[tp])],
    "False Negative (fake, missed)":     fn[np.argmin(proba_fake[fn])],
    "False Positive (real, flagged)":    fp[np.argmax(proba_fake[fp])],
}

### 12.1 SP-LIME: Choosing Representative Articles

LIME explains one article at a time, so the articles we look at matter. Instead of choosing them by hand, SP-LIME (submodular pick) generates LIME explanations for a pool of 200 test articles and then picks the 5 whose explanations together cover as many of the important words as possible, with as little repetition as possible. The pick is based on the words in the explanations, not on the article's category or whether it is fake, so after the pick we check which categories, labels and outcomes were selected.

In [ ]:
N_CANDIDATES = 200
N_PICKS = 5

rng_sp = np.random.RandomState(RANDOM_STATE)
cand_idx = rng_sp.choice(len(test_text), size=N_CANDIDATES, replace=False)
cand_text = test_text.iloc[cand_idx].tolist()

sp = SubmodularPick(
    explainer_lime, cand_text, predict_proba_text,
    method="full",
    num_exps_desired=N_PICKS,
    num_features=7,
    num_samples=1000,
    top_labels=None          # explain the fake class (label 1) for every article
)
sp_idx = cand_idx[sp.V]      # positions of the picked articles in the test set

In [ ]:
def outcome(i):
    t, p = int(y_true_arr[i]), int(y_pred_arr[i])
    return {(1, 1): "TP (fake, caught)", (1, 0): "FN (fake, missed)",
            (0, 1): "FP (real, flagged)", (0, 0): "TN (real, correct)"}[(t, p)]

sp_table = pd.DataFrame({
    "category": [df.loc[y_test.index[i], "category"] for i in sp_idx],
    "true_label": ["fake" if y_true_arr[i] == 1 else "real" for i in sp_idx],
    "P(fake)": [round(float(proba_fake[i]), 3) for i in sp_idx],
    "outcome": [outcome(i) for i in sp_idx],
})
display(sp_table)

In [ ]:
for rank, (exp, i) in enumerate(zip(sp.sp_explanations, sp_idx), start=1):
    print(f"SP-LIME pick {rank} | P(fake) = {proba_fake[i]:.3f}")
    exp.as_pyplot_figure(label=1)
    plt.tight_layout()
    plt.show()

### 12.2 LIME: Word-Removal Sanity Checks

LIME removes words from one article and watches how the predicted probability changes. Three cases are inspected: a correctly caught fake article (TP), a missed fake article (FN), and a real article wrongly flagged as fake (FP). Weights are for the fake class: positive = pushes toward fake.

In [ ]:
for name, i in cases.items():
    exp = explainer_lime.explain_instance(
        test_text.iloc[i], predict_proba_text,
        num_features=7, num_samples=1000, labels=(1,)
    )
    print(f"{name} | P(fake) = {proba_fake[i]:.3f}")
    exp.as_pyplot_figure(label=1)
    plt.tight_layout()
    plt.show()

### 12.3 SHAP Waterfall (one article)

LIME explains one prediction by approximating the model around that article, whereas SHAP splits the model's own prediction into a starting value plus one contribution per term that adds up exactly to the model's final output, measured in log-odds, so the waterfall gives an exact term-by-term breakdown for one example article.

In [ ]:
fn_idx = cases["False Negative (fake, missed)"]
x_fn = X_test_dense[fn_idx:fn_idx + 1]

sv_fn = explainer_shap.shap_values(x_fn, check_additivity=False)
if isinstance(sv_fn, list):
    sv_fn_fake = sv_fn[1]
elif sv_fn.ndim == 3:
    sv_fn_fake = sv_fn[:, :, 1]
else:
    sv_fn_fake = sv_fn

shap.plots.waterfall(
    shap.Explanation(
        values=sv_fn_fake[0], base_values=base_value,
        data=x_fn[0], feature_names=list(feature_names)
    ),
    max_display=7
)

## 13. Summary